# PySpark – Customer Dataset

**Goal:** Read `customer.csv`, see how Spark divides the 100 records into partitions, calculate sales, and find total sales per product.

Keep `customer.csv` in the same folder as this notebook.

## 1. Install and start Spark

Install PySpark once if it is not already installed. `SparkSession` starts our Spark application. `local[*]` lets Spark use the available CPU cores on this computer.

In [12]:
#!pip install pyspark

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum

spark = SparkSession.builder.appName("CustomerSales").master("local[*]").getOrCreate()

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Load the CSV

`header=True` uses the first row as column names. `inferSchema=True` detects data types. The CSV has an extra `Unnamed: 0` index column, so we drop it.

In [15]:
df = spark.read.csv("customer.csv", header=True, inferSchema=True).drop("Unnamed: 0")
df.show(5)
print("Total records:", df.count())

+---+---+--------+--------+---+-----+
|_c0| ID|Customer| Product|Qty|Price|
+---+---+--------+--------+---+-----+
|  0|  1|    C101|  Laptop|  1|50000|
|  1|  2|    C102|  Tablet|  1|20000|
|  2|  3|    C103|  Mobile|  1|15000|
|  3|  4|    C104|  Laptop|  3|50000|
|  4|  5|    C105|Keyboard|  1| 1500|
+---+---+--------+--------+---+-----+
only showing top 5 rows
Total records: 100


## 3. Divide records into partitions

A **partition** is a smaller part of the data that Spark can process as a task. Here, `repartition(4)` redistributes the records into 4 partitions. Spark may move data during this step (shuffle).

The next line shows how many records are in each partition. The counts may not be equal.

In [16]:
df4 = df.repartition(4)
print("Number of partitions:", df4.rdd.getNumPartitions())
print("Records in each partition:", df4.rdd.mapPartitions(lambda rows: [sum(1 for _ in rows)]).collect())

Number of partitions: 4
Records in each partition: [25, 25, 25, 25]


In [17]:
# A simpler way to count records in each partition
counts = df4.rdd.mapPartitions(lambda it: [len(list(it))]).collect()
print("Records in each partition:", counts)

Records in each partition: [25, 25, 25, 25]


## 4. Calculate sales

**Sales = Quantity × Price**

`withColumn()` creates a new column. `col()` refers to a column in the DataFrame. `show()` displays the result and triggers Spark to execute the work.

In [18]:
sales = df4.withColumn("Sales", col("Qty") * col("Price"))
sales.select("ID", "Product", "Qty", "Price", "Sales").show(10)

+---+-------+---+-----+------+
| ID|Product|Qty|Price| Sales|
+---+-------+---+-----+------+
| 15| Laptop|  1|50000| 50000|
| 62| Laptop|  1|50000| 50000|
| 92|  Mouse|  1|  500|   500|
| 86| Tablet|  2|20000| 40000|
| 71| Laptop|  3|50000|150000|
|  7| Laptop|  1|50000| 50000|
| 34| Tablet|  2|20000| 40000|
| 94| Mobile|  3|15000| 45000|
| 98|  Mouse|  1|  500|   500|
| 26| Laptop|  3|50000|150000|
+---+-------+---+-----+------+
only showing top 10 rows


## 5. Find total sales for each product

`groupBy()` groups records with the same product. `sum()` adds their sales. Grouping may require a **shuffle**, where Spark moves rows so matching products can be combined. The final table shows total quantity and total sales for each product.

In [19]:
result = sales.groupBy("Product").agg(
    spark_sum("Qty").alias("Total_Quantity"),
    spark_sum("Sales").alias("Total_Sales")
)
result.show()

+--------+--------------+-----------+
| Product|Total_Quantity|Total_Sales|
+--------+--------------+-----------+
|  Laptop|            43|    2150000|
|   Mouse|            26|      13000|
|  Mobile|            51|     765000|
|  Tablet|            45|     900000|
|Keyboard|            37|      55500|
+--------+--------------+-----------+



## 7. Stop Spark

In [20]:
spark.stop()